In [8]:
import os
import keras

import tensorflow as tf

model_path = "finetuned_model_mix_v2.keras"

try:
    model = keras.models.load_model(model_path, compile=False)
    print("✅ 模型載入成功！")
    
    # 順手找出你的 Grad-CAM 產圖關鍵層
    for layer in reversed(model.layers):
        if 'conv' in layer.name.lower():
            print(f"💡 你的 Grad-CAM 目標卷積層是：{layer.name}")
            break
except Exception as e:
    print(f"❌ 發生錯誤: {e}")

✅ 模型載入成功！
💡 你的 Grad-CAM 目標卷積層是：conv2d_38


In [15]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ data (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 111, 111,  │        288 │ data[0][0]        │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 111, 111,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 109, 109,  │      9,216 │ activation_1[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 109, 109,  │     18,432 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 54, 54,    │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 26, 26,    │     46,080 │ max_pooling2d_1[… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 26, 26,    │        320 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 26, 26,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 24, 24,    │    138,240 │ activation_4[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 24,    │        768 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 24, 24,    │          0 │ batch_normalizat

 Total params: 5,503,334 (20.99 MB)

 Trainable params: 504,966 (1.93 MB)

 Non-trainable params: 4,998,368 (19.07 MB)

In [ ]:
import os
import cv2
import numpy as np
import keras
import tensorflow as tf
import matplotlib.pyplot as plt

# ===================== 1. 載入模型 =====================
model = keras.models.load_model(
    "finetuned_model_mix_v2.keras",
    compile=False
)

print("✅ 模型載入成功")

# ===================== 2. 自動找候選 layer =====================
def get_candidate_layers(model):
    candidates = []
    for layer in model.layers:
        if isinstance(layer, keras.layers.Conv2D):
            candidates.append(layer.name)
        if "activation" in layer.name.lower():
            candidates.append(layer.name)
    return candidates[-8:]  # 只取最後幾層（重要）

candidate_layers = get_candidate_layers(model)
print("🎯 候選 layers:", candidate_layers)

# ===================== 3. 前處理 =====================
IMG_SIZE = 224

def preprocess(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    return np.expand_dims(img, axis=0), img

# ===================== 4. Grad-CAM =====================
def make_gradcam(img_array, model, layer_name):
    grad_model = keras.models.Model(
        model.inputs,
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array)
        class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)

    if grads is None:
        return None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    # 🔥 核心修正（避免藍圖）
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)

    if tf.reduce_max(heatmap) == 0:
        return None

    heatmap /= tf.reduce_max(heatmap)

    return heatmap.numpy()


# ===================== 5. 自動選最佳 layer =====================
def find_best_layer(img_array, model, candidates):
    best_layer = None
    best_score = -1
    best_heatmap = None

    for layer in candidates:
        heatmap = make_gradcam(img_array, model, layer)
        if heatmap is None:
            continue

        score = np.std(heatmap)  # 🔥 關鍵：變化越大越好

        print(f"{layer} → score: {score:.4f}")

        if score > best_score:
            best_score = score
            best_layer = layer
            best_heatmap = heatmap

    return best_layer, best_heatmap

# ===================== 6. 疊圖 =====================
def overlay(img, heatmap):
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img_bgr = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

    result = cv2.addWeighted(heatmap_color, 0.4, img_bgr, 0.6, 0)
    return result

# ===================== 7. 主程式 =====================
img_path = "emotion_dataset/Delight/Yo_Delight_25_orig.jpg"

img_array, original_img = preprocess(img_path)

best_layer, heatmap = find_best_layer(img_array, model, candidate_layers)

print(f"\n🏆 最佳 Grad-CAM layer: {best_layer}")

result = overlay(original_img, heatmap)

# ===================== 8. 顯示 =====================
plt.figure(figsize=(10,4))

plt.subplot(1,3,1)
plt.title("Original")
plt.imshow(original_img)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Grad-CAM")
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.show()

✅ 模型載入成功
🎯 候選 layers: ['conv2d_36', 'activation_36', 'conv2d_34', 'conv2d_37', 'activation_34', 'activation_37', 'conv2d_38', 'activation_38']


c:\Users\User\anaconda3\envs\xai_lab\lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['data']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


ValueError: Exception encountered when calling Functional.call().

[1mInput 0 of layer "conv2d_1" is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (1, 224, 224, 3)[0m

Arguments received by Functional.call():
  • inputs=array([[[[0.07843137, 0.05882353, 0.04705882],
         [0.05490196, 0.05098039, 0.04313725],
         [0.04705882, 0.05098039, 0.05882353],
         ...,
         [0.46666667, 0.40784314, 0.31764706],
         [0.51764706, 0.45882353, 0.36470588],
         [0.5372549 , 0.50588235, 0.49411765]],

        [[0.07058824, 0.0627451 , 0.05490196],
         [0.07058824, 0.06666667, 0.05490196],
         [0.08235294, 0.0745098 , 0.07843137],
         ...,
         [0.4627451 , 0.41568627, 0.32941176],
         [0.50588235, 0.4627451 , 0.41960784],
         [0.54901961, 0.53333333, 0.53333333]],

        [[0.05882353, 0.05882353, 0.06666667],
         [0.0627451 , 0.04705882, 0.05098039],
         [0.09019608, 0.08

In [10]:
import os
import cv2
import numpy as np
import keras
import tensorflow as tf 
import matplotlib.pyplot as plt

# ===================== 1. 路徑與模型設定 =====================
MODEL_PATH = "finetuned_model_mix_v2.keras"
DATASET_DIR = "emotion_dataset"             
OUTPUT_DIR = "results/gradcam_outputs"      
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 依據你的發現，使用原生 keras 載入
model = keras.models.load_model(MODEL_PATH, compile=False)
TARGET_LAYER = "conv2d_38" 

# ===================== 2. 影像預處理函數 =====================
IMG_W, IMG_H = 224, 224
HAAR_DIR = cv2.data.haarcascades
FACE_CASCADE = cv2.CascadeClassifier(os.path.join(HAAR_DIR, "haarcascade_frontalface_default.xml"))

def preprocess_local_image(img_path):
    frame_bgr = cv2.imread(img_path)
    if frame_bgr is None: return None, None
    
    frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    results = FACE_CASCADE.detectMultiScale(frame_gray, 1.1, 5, minSize=(60,60))
    
    if len(results) == 0: return None, None

    x, y, w, h = max(results, key=lambda b: b[2]*b[3])
    roi = frame_gray[y:y+h, x:x+w]
    
    resized = cv2.resize(roi, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    
    # 確保輸入符合 (1, 224, 224, 1) 並進行正規化
    face_input = resized.astype(np.float32)[..., None] / 255.0
    face_input = np.expand_dims(face_input, axis=0)
    
    return face_input, resized

# ===================== 3. 批次執行 Grad-CAM =====================
def run_gradcam_experiment():
    # 改進：直接取得倒數第二層的輸出（Logits），避免 Softmax 讓梯度消失
    # 假設最後一層是 'dense_1'，我們建立一個能輸出最後卷積層與 Logits 的模型
    grad_model = keras.models.Model(
        model.inputs, [model.get_layer(TARGET_LAYER).output, model.layers[-1].output]
    )

    for emotion in os.listdir(DATASET_DIR):
        emotion_path = os.path.join(DATASET_DIR, emotion)
        if not os.path.isdir(emotion_path): continue
        
        save_path = os.path.join(OUTPUT_DIR, emotion)
        os.makedirs(save_path, exist_ok=True)
        
        print(f"正在分析情緒類別：{emotion}...")
        
        for filename in os.listdir(emotion_path):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            
            img_input, face_roi = preprocess_local_image(os.path.join(emotion_path, filename))
            if img_input is None: continue

            # 使用 GradientTape 計算梯度
            with tf.GradientTape() as tape:
                conv_outputs, predictions = grad_model(img_input)
                class_idx = np.argmax(predictions[0])
                loss = predictions[:, class_idx]

            # 抓取卷積層相對於損失的梯度
            grads = tape.gradient(loss, conv_outputs)
            
            # 全域平均池化梯度 (Importance weights)
            pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
            
            # 將卷積層輸出與權重加權求和
            conv_outputs = conv_outputs[0]
            heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
            heatmap = tf.squeeze(heatmap)

            # --- 強化熱區顯示的關鍵步驟 ---
            # 1. 使用 ReLU 只保留正貢獻區域
            heatmap = np.maximum(heatmap, 0)
            # 2. 正規化至 0~1，並加上極小值防呆
            heatmap /= (np.max(heatmap) + 1e-10)

            # 視覺化疊加
            heatmap_resized = cv2.resize(heatmap, (face_roi.shape[1], face_roi.shape[0]))
            heatmap_u8 = np.uint8(255 * heatmap_resized)
            
            # 使用 JET 色譜：強烈特徵會顯示為紅色，弱特徵為藍色
            heatmap_color = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
            
            # 將底圖轉為 BGR 並疊加熱區 (alpha=0.6 代表底圖權重)
            face_bgr = cv2.cvtColor(face_roi, cv2.COLOR_GRAY2BGR)
            overlay = cv2.addWeighted(face_bgr, 0.6, heatmap_color, 0.4, 0)
            
            # 儲存結果
            cv2.imwrite(os.path.join(save_path, f"cam_{filename}"), overlay)

run_gradcam_experiment()
print("🎉 Grad-CAM 產出完成")

正在分析情緒類別：Boredom...
正在分析情緒類別：Confusion...
正在分析情緒類別：Delight...
正在分析情緒類別：Engagement...
正在分析情緒類別：Frustration...
正在分析情緒類別：Suprise...
🎉 Grad-CAM 產出完成！現在你的圖片應該會有明顯的紅、黃色區塊了。
